# Query the InsuranceAgent run database

This notebook is for exploring the SQLite data stored under `data/runs.db`.
It helps you inspect persisted runs, failures, checkpoints, and answer-bank entries from the browser agent.

Use the cells below as a playground: run a query, change a parameter, and inspect the result.

In [ ]:
import json
import sqlite3
from pathlib import Path
from pprint import pprint

import pandas as pd

DB_PATH = Path('data/runs.db')
print(f'DB exists: {DB_PATH.exists()}')
print(f'Full path: {DB_PATH.resolve()}')

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH.resolve()}.")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)


def first_row(sql: str, params: tuple = ()):
    row = conn.execute(sql, params).fetchone()
    return dict(row) if row else None

print('Connected to runs.db')
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"))

## 1. Import Required Libraries

This section imports the libraries needed for database access, JSON inspection, and friendly tabular output.

In [ ]:
import sqlite3
from pathlib import Path
import json
import pandas as pd

DB_PATH = Path('data/runs.db')
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)

print('Ready to query SQLite')
print(f'Connected to: {DB_PATH.resolve()}')

## 2. Set Up the Database Path and Connection

The project stores its run history in a SQLite file under the `data` folder. You can change the path here if needed.

In [ ]:
DB_PATH = Path('data/runs.db')
print(f'Using {DB_PATH.resolve()}')

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)

# Show tables
q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

## 3. Define a Query Function

This reusable helper keeps the notebook interactive and easy to extend.

In [ ]:

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)


def pretty_json(value):
    
    if value is None:
        return None
    return json.loads(value) if isinstance(value, str) else value

print('Helper ready')

## 4. Run a Few Sample Queries

These examples show the basic shape of the persisted data.

In [ ]:
# List all tables
run_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

In [ ]:
# Recent runs
run_query(
    """
    SELECT id, provider_id, goal, status, mode, llm_calls, error, created_at, updated_at
    FROM runs
    ORDER BY created_at DESC
    LIMIT 20
    """
)

In [ ]:
# Failed runs only
run_query(
    """
    SELECT id, provider_id, goal, status, llm_calls, error, created_at
    FROM runs
    WHERE status = 'failed'
    ORDER BY created_at DESC
    LIMIT 20
    """
)

In [ ]:
# Pending checkpoints waiting for a human
run_query(
    """
    SELECT id, run_id, provider_id, goal, reason, fingerprint, question, status, created_at
    FROM checkpoints
    WHERE status = 'pending'
    ORDER BY created_at DESC
    LIMIT 20
    """
)

In [ ]:
# Reusable answers stored in the answer bank
run_query(
    """
    SELECT fingerprint, provider_id, field_key, value, question, profile_key, use_count
    FROM answers
    ORDER BY use_count DESC, updated_at DESC
    LIMIT 20
    """
)

## 5. Add Interactive Controls for Experimentation

Use a run id or status filter to inspect a particular run and investigate its stored state.

In [ ]:
run_id = ''  # set this to a run id you want to inspect, e.g. 'e687340fb479'
status_filter = 'failed'  # try: 'failed', 'awaiting_human', 'completed', 'running'
provider_filter = ''  # optional, e.g. 'forsikringsguiden'

if run_id:
    display(run_query(
        "SELECT * FROM runs WHERE id = ?",
        (run_id,)
    ))
else:
    display(run_query(
        """
        SELECT id, provider_id, goal, status, llm_calls, error, created_at
        FROM runs
        WHERE (? = '' OR status = ?) AND (? = '' OR provider_id = ?)
        ORDER BY created_at DESC
        LIMIT 20
        """,
        (status_filter, status_filter, provider_filter, provider_filter)
    ))

## 6. Inspect and Format Responses

This section decodes the JSON blobs that store the full run state and the result payload.

In [ ]:
# Inspect one run in detail by setting run_id above
run_id = run_id if 'run_id' in globals() else ''

if not run_id:
    print('Set run_id above to a specific run, or use the previous cell to choose a result.')
else:
    row = conn.execute("SELECT * FROM runs WHERE id = ?", (run_id,)).fetchone()
    if row is None:
        print('No such run found')
    else:
        state = json.loads(row['state'])
        print('RUN SUMMARY')
        print('id:', row['id'])
        print('provider:', row['provider_id'])
        print('goal:', row['goal'])
        print('status:', row['status'])
        print('error:', row['error'])
        print('llm_calls:', row['llm_calls'])
        print('last_url:', state.get('last_url'))
        print('answers:', state.get('answers'))
        print('pending_checkpoint:', state.get('pending_checkpoint'))
        print('\nTRAJECTORY COUNT:', len(state.get('trajectory', [])))
        print('FIRST 3 STEPS:')
        for step in state.get('trajectory', [])[:3]:
            print(json.dumps(step, indent=2, ensure_ascii=False))

In [ ]:
# Show all results rows for a provider/goal combo
provider = 'forsikringsguiden'
goal = 'bilforsikring'

run_query(
    """
    SELECT run_id, provider_id, goal, payload, created_at
    FROM results
    WHERE provider_id = ? AND goal = ?
    ORDER BY created_at DESC
    LIMIT 10
    """,
    (provider, goal)
)

## 7. Handle Errors and Edge Cases

This protects the notebook from bad inputs, empty tables, and missing values.

In [ ]:
# Safe helpers for edge cases

def safe_table(sql: str, params: tuple = (), default='No rows found'):
    df = run_query(sql, params)
    return df if not df.empty else print(default)


def safe_run_state(run_id: str):
    row = conn.execute("SELECT state FROM runs WHERE id = ?", (run_id,)).fetchone()
    if row is None:
        return None
    return json.loads(row['state'])


try:
    safe_table("SELECT * FROM runs LIMIT 1", default='No runs in the database yet')
except Exception as exc:
    print(f'Error while querying runs table: {exc}')

try:
    state = safe_run_state('not-a-real-run-id')
    print('Safe run-state check:', state)
except Exception as exc:
    print(f'Unexpected error: {exc}')

In [ ]:
# Your playground: write your own query below and run it.
# Example:
# run_query("SELECT * FROM runs WHERE status = 'failed' LIMIT 10")

# 1) Try a status filter
# run_query("SELECT id, status, provider_id, goal FROM runs WHERE status = 'failed' ORDER BY created_at DESC LIMIT 20")

# 2) Try filtering by provider
# run_query("SELECT id, goal, status, llm_calls FROM runs WHERE provider_id = 'forsikringsguiden' ORDER BY created_at DESC LIMIT 20")

# 3) Try inspecting checkpoints
# run_query("SELECT * FROM checkpoints ORDER BY created_at DESC LIMIT 20")

print('Notebook is ready for custom queries.')